# P6 — Binance Backtest Analysis

Visual analysis of 24 strategy×pair backtests on 3 years of Binance data.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

RESULTS = Path("../results")

benchmarks = json.loads((RESULTS / "P6_benchmarks.json").read_text())
phase_d = json.loads((RESULTS / "P6_phase_d_results.json").read_text())
survivors = json.loads((RESULTS / "P6_phase_e_survivors.json").read_text())
walkforward = json.loads((RESULTS / "P6_phase_f_walkforward.json").read_text()) if (RESULTS / "P6_phase_f_walkforward.json").exists() else {}

print(f"Phase D results: {len(phase_d)} combinations")
print(f"Survivors: {len(survivors)}")
print(f"Walk-forward: {len(walkforward)}")

## 1. Data Coverage Heatmap

In [ ]:
# Parse coverage from P6_data_coverage.md
coverage_path = RESULTS / "P6_data_coverage.md"
coverage_data = []
if coverage_path.exists():
    in_table = False
    for line in coverage_path.read_text().split("\n"):
        if "P6 Period Coverage" in line:
            in_table = True
            continue
        if in_table and line.startswith("|") and "Pair" not in line and "---" not in line:
            parts = [p.strip() for p in line.split("|")[1:-1]]
            if len(parts) >= 5:
                pair, tf, expected, actual, cov_pct = parts[:5]
                coverage_data.append({
                    "Pair": pair, "TF": tf,
                    "Coverage %": float(cov_pct.replace("%", ""))
                })
        if in_table and line.startswith("## 3"):
            break

if coverage_data:
    df_cov = pd.DataFrame(coverage_data)
    pivot = df_cov.pivot(index="Pair", columns="TF", values="Coverage %")
    # Reorder TF columns
    tf_order = ["5m", "15m", "1h", "4h", "1d", "1w"]
    pivot = pivot[[c for c in tf_order if c in pivot.columns]]

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn", vmin=70, vmax=100,
                linewidths=1, ax=ax, cbar_kws={"label": "Coverage %"})
    ax.set_title("Data Coverage — P6 Period (2023-04 to 2026-04)")
    plt.tight_layout()
    plt.show()
else:
    print("No coverage data found.")

## 2. Benchmarks Comparison

In [ ]:
pairs = ["BTC/USDC", "ETH/USDC", "SOL/USDC"]
bm_rows = []
for bm_type, label in [("buy_and_hold", "Buy & Hold"), ("dca_fixed_15usd_weekly", "DCA $15/wk")]:
    for pair in pairs:
        m = benchmarks.get(bm_type, {}).get(pair, {})
        if "error" not in m:
            bm_rows.append({
                "Pair": pair, "Strategy": label,
                "Return %": m.get("total_return_pct", 0),
                "Sharpe": m.get("sharpe_ratio", 0),
                "MaxDD %": m.get("max_drawdown_pct", 0),
            })

df_bm = pd.DataFrame(bm_rows)
if not df_bm.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.barplot(data=df_bm, x="Pair", y="Return %", hue="Strategy", ax=axes[0])
    axes[0].set_title("Benchmark Returns (3yr)")
    axes[0].axhline(y=0, color="black", linewidth=0.5)
    
    sns.barplot(data=df_bm, x="Pair", y="Sharpe", hue="Strategy", ax=axes[1])
    axes[1].set_title("Benchmark Sharpe Ratios")
    axes[1].axhline(y=1.0, color="red", linestyle="--", linewidth=0.5, label="Threshold")
    
    plt.tight_layout()
    plt.show()
else:
    print("No benchmark data.")

## 3. First Pass Overview — Sharpe vs Max Drawdown

In [ ]:
rows = []
for key, data in phase_d.items():
    if "error" in data:
        continue
    test = data.get("test", {})
    rows.append({
        "Strategy": data["strategy"].replace("grok_", "").replace("gemini_", ""),
        "Pair": data["pair"],
        "Sharpe (test)": test.get("sharpe_ratio", 0),
        "MaxDD % (test)": test.get("max_drawdown_pct", 0),
        "Return % (test)": test.get("total_return_pct", 0),
        "Survivor": key in survivors,
    })

df = pd.DataFrame(rows)
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    markers = {"BTC/USDC": "o", "ETH/USDC": "s", "SOL/USDC": "D"}
    strategies = df["Strategy"].unique()
    colors = sns.color_palette("husl", len(strategies))
    color_map = dict(zip(strategies, colors))
    
    for _, row in df.iterrows():
        ax.scatter(
            row["MaxDD % (test)"], row["Sharpe (test)"],
            c=[color_map[row["Strategy"]]],
            marker=markers.get(row["Pair"], "o"),
            s=200 if row["Survivor"] else 80,
            alpha=1.0 if row["Survivor"] else 0.4,
            edgecolors="black" if row["Survivor"] else "none",
            linewidths=2 if row["Survivor"] else 0,
        )
    
    # Threshold lines
    ax.axhline(y=1.0, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Sharpe=1.0")
    ax.axvline(x=25, color="red", linestyle="--", linewidth=1, alpha=0.5, label="MaxDD=25%")
    
    # Legend for strategies
    for s, c in color_map.items():
        ax.scatter([], [], c=[c], s=80, label=s)
    # Legend for pairs
    for p, m in markers.items():
        ax.scatter([], [], c="gray", marker=m, s=80, label=p)
    
    ax.set_xlabel("Max Drawdown % (test)")
    ax.set_ylabel("Sharpe Ratio (test)")
    ax.set_title("First Pass — Sharpe vs Max Drawdown (survivors have black border)")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No Phase D data.")

## 4. First Pass — Return Heatmap (Strategy × Pair)

In [ ]:
if not df.empty:
    pivot_ret = df.pivot(index="Strategy", columns="Pair", values="Return % (test)")
    pivot_sharpe = df.pivot(index="Strategy", columns="Pair", values="Sharpe (test)")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sns.heatmap(pivot_ret, annot=True, fmt="+.1f", cmap="RdYlGn", center=0,
                linewidths=1, ax=axes[0])
    axes[0].set_title("Test Return % by Strategy × Pair")
    
    sns.heatmap(pivot_sharpe, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
                linewidths=1, ax=axes[1])
    axes[1].set_title("Test Sharpe by Strategy × Pair")
    
    plt.tight_layout()
    plt.show()

## 5. Walk-Forward Stability (Sharpe Boxplots)

In [ ]:
if walkforward:
    wf_rows = []
    for key, wf in walkforward.items():
        label = f"{wf['strategy'].replace('grok_', '').replace('gemini_', '')}\n{wf['pair']}"
        for w in wf.get("windows", []):
            if "metrics" in w:
                wf_rows.append({
                    "Combo": label,
                    "Sharpe": w["metrics"]["sharpe_ratio"],
                    "Return %": w["metrics"]["total_return_pct"],
                })
    
    if wf_rows:
        df_wf = pd.DataFrame(wf_rows)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        sns.boxplot(data=df_wf, x="Combo", y="Sharpe", ax=axes[0])
        axes[0].axhline(y=0, color="red", linestyle="--", linewidth=1)
        axes[0].set_title("Walk-Forward Sharpe Distribution")
        axes[0].tick_params(axis="x", rotation=45)
        
        sns.boxplot(data=df_wf, x="Combo", y="Return %", ax=axes[1])
        axes[1].axhline(y=0, color="red", linestyle="--", linewidth=1)
        axes[1].set_title("Walk-Forward Return Distribution")
        axes[1].tick_params(axis="x", rotation=45)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No walk-forward window data.")
else:
    print("No walk-forward results.")

## 6. Walk-Forward Consistency Scores

In [ ]:
if walkforward:
    cons_rows = []
    for key, wf in walkforward.items():
        label = f"{wf['strategy'].replace('grok_', '').replace('gemini_', '')}\n{wf['pair']}"
        cons_rows.append({
            "Combo": label,
            "Consistency": wf.get("consistency_score", 0) * 100,
            "Mean Sharpe": wf.get("mean_metrics", {}).get("sharpe_ratio", 0),
        })
    
    df_cons = pd.DataFrame(cons_rows)
    if not df_cons.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.barh(df_cons["Combo"], df_cons["Consistency"])
        ax.axvline(x=50, color="red", linestyle="--", linewidth=1, label="50% threshold")
        ax.set_xlabel("Consistency Score (% positive Sharpe windows)")
        ax.set_title("Walk-Forward Consistency")
        ax.legend()
        
        for bar, val in zip(bars, df_cons["Consistency"]):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{val:.0f}%", va="center")
        
        plt.tight_layout()
        plt.show()

## 7. Train vs Test Comparison

In [ ]:
tt_rows = []
for key, data in phase_d.items():
    if "error" in data:
        continue
    train = data.get("train", {})
    test = data.get("test", {})
    label = f"{data['strategy'].replace('grok_', '').replace('gemini_', '')}\n{data['pair']}"
    tt_rows.append({
        "Combo": label,
        "Train Sharpe": train.get("sharpe_ratio", 0),
        "Test Sharpe": test.get("sharpe_ratio", 0),
        "Survivor": key in survivors,
    })

df_tt = pd.DataFrame(tt_rows)
if not df_tt.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    max_val = max(df_tt["Train Sharpe"].max(), df_tt["Test Sharpe"].max(), 2) * 1.1
    min_val = min(df_tt["Train Sharpe"].min(), df_tt["Test Sharpe"].min(), -1) * 1.1
    
    ax.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.3, label="y=x (no overfit)")
    
    for _, row in df_tt.iterrows():
        color = "green" if row["Survivor"] else "red"
        alpha = 1.0 if row["Survivor"] else 0.4
        ax.scatter(row["Train Sharpe"], row["Test Sharpe"], c=color, s=60, alpha=alpha)
        if row["Survivor"]:
            ax.annotate(row["Combo"], (row["Train Sharpe"], row["Test Sharpe"]),
                        fontsize=7, ha="left", va="bottom")
    
    ax.set_xlabel("Train Sharpe")
    ax.set_ylabel("Test Sharpe")
    ax.set_title("Train vs Test Sharpe (green = survivor)")
    ax.axhline(y=1.0, color="blue", linestyle=":", alpha=0.3)
    ax.axvline(x=1.0, color="blue", linestyle=":", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Final Recommendations

In [ ]:
print("=" * 60)
print("  FINAL RECOMMENDATIONS")
print("=" * 60)

if walkforward:
    for key, wf in sorted(walkforward.items(), key=lambda x: -x[1].get("consistency_score", 0)):
        cs = wf.get("consistency_score", 0)
        mm = wf.get("mean_metrics", {})
        priority = "P1" if cs >= 0.625 else ("P2" if cs >= 0.5 else "RISK")
        print(f"  [{priority}] {wf['strategy']:40s} {wf['pair']:10s} "
              f"consistency={cs:.0%} sharpe={mm.get('sharpe_ratio', 0):.2f}")
elif survivors:
    print("  Walk-forward not yet run. Survivors from Phase E:")
    for key, s in survivors.items():
        test = s.get("test", {})
        print(f"  {s['strategy']:40s} {s['pair']:10s} "
              f"sharpe={test.get('sharpe_ratio', 0):.2f}")
else:
    print("  No survivors. All combinations failed acceptance criteria.")
    print("  Consider relaxing thresholds or investigating strategy bugs.")